In [2]:
import os
import numpy as np
import pandas as pd
import pickle
import random

In [29]:
FIGURES_PATH = "figures"
os.makedirs(FIGURES_PATH,exist_ok=True)
# Set datafiles folder.
datafolder = "datafiles"
# Create directory in case it does not exist.
os.makedirs(datafolder,exist_ok=True)

# Set the name of the datafile.
datafile = "Dataset.npy"
# Load the dataset.
dataset = np.load(datafile)
# Define the spliter lambda function in order to tokenize the initial string 
# data.
spliter = lambda s: s.split(",")
# Apply the spliter function at each element of the dataset string array.
dataset = np.array([spliter(x) for x in dataset])

# Set the pickle file for storing the initial dataframe.
pickle_file = os.path.join(datafolder,"dataframe.pkl")
# Check the existence of the specifiied file.
#read pickled datafile
if os.path.exists(pickle_file):
    # Load the pickle file.
    dataframe = pd.read_pickle(pickle_file)
else:
    # Create the dataframe object.
    dataframe = pd.DataFrame(dataset, columns=["user","item","rating","date"])
    # Convert the string elements of the "users" series into integers.
    dataframe["user"] = dataframe["user"].apply(lambda s:np.int64(s.replace("ur","")))
    # Convert the string elements of the "items" series into integers.
    dataframe["item"] = dataframe["item"].apply(lambda s:np.int64(s.replace("tt","")))
    # Conver the string elements of the "ratings" series into integers.
    dataframe["rating"] = dataframe["rating"].apply(lambda s:np.int64(s))
    # Convert the string elements of the "dates" series into datetime objects.
    dataframe["date"] = pd.to_datetime(dataframe["date"])
    dataframe.to_pickle(pickle_file)

In [30]:
#find and remove duplicates and remove date column
duplicates=dataframe.duplicated().sum()
if (duplicates>0):
    print("Number of duplicate values:",duplicates)
    dataframe.drop_duplicates(inplace=True)
dataframe.drop(columns=["date"],inplace=True)

dataframe.to_pickle(pickle_file)

dataframe

Number of duplicate values: 9382


,user,item,rating
0,4592644,120884,10
1,3174947,118688,3
2,3780035,387887,8
3,4592628,346491,1
4,3174947,94721,8
...,...,...,...
4669815,581842,107977,6
4669816,3174947,103776,8
4669817,4592639,107423,9
4669818,4581944,102614,8


In [31]:

def get_stats(dataframe):
    # Get the unique users in the dataset.
    users = dataframe["user"].unique()
    # Get the number of unique users.
    users_num = len(users)
    # Get the unique items in the dataset.
    items = dataframe["item"].unique()
    items_num = len(items)
    # Get the total number of existing ratings.
    ratings_num = dataframe.shape[0]
    # Report the number of unique users and items in the dataset.
    print("DATASET: {0} number of unique users and {1} of unique items".format(users_num,items_num))
    # Report the total number of existing ratings in the dataset.
    print("DATASET: {} total number of existing ratings".format(ratings_num))
    return users_num,items_num,ratings_num
users_num,items_num,ratings_num = get_stats(dataframe)

DATASET: 1499238 number of unique users and 351109 of unique items
DATASET: 4660438 total number of existing ratings


In [44]:
#Now we  need to choose a subset of the ratings in the dataset that corresponded to users and items within a specific range of ratings.
#We also need to ensure that by the end of the process there are not any users that have rated too few items or items that have been rated too few times.

In [32]:
minimum_user_ratings = 100
maximum_user_ratings = 300
minimum_item_ratings = 10

new_df = dataframe.copy()
# Initial filtering
def filter_users_and_items(df):
    # Filter users
    user_ratings_count = df.groupby("user")["rating"].count().sort_values(ascending=
                                    False).reset_index(name="ratings_num")
    filtered_users = user_ratings_count.loc[(user_ratings_count["ratings_num"] >= minimum_user_ratings) & 
                                            (user_ratings_count["ratings_num"] <= maximum_user_ratings)]
    df = df.loc[df["user"].isin(filtered_users["user"])]
    print("Filtered users: {} -> {}".format(user_ratings_count.shape[0], filtered_users.shape[0]))

    # Filter items
    item_ratings_count = df.groupby("item")["rating"].count().sort_values(ascending=
                                    False).reset_index(name="users_rated")
    filtered_items = item_ratings_count.loc[item_ratings_count["users_rated"] >= minimum_item_ratings]
    df = df.loc[df["item"].isin(filtered_items["item"])]
    print("Filtered items: {} -> {}".format(item_ratings_count.shape[0], filtered_items.shape[0]))

    return df

# Iteratively filter users and items until no more changes
previous_shape = None
current_shape = new_df.shape

while previous_shape != current_shape:
    previous_shape = current_shape
    new_df = filter_users_and_items(   new_df)
    current_shape = new_df.shape

# Reset index and drop the old index column
final_df = new_df.reset_index(drop=True)

# Get final stats
users_num, items_num, ratings_num = get_stats(final_df)



Filtered users: 1499238 -> 2290
Filtered items: 91878 -> 7078
Filtered users: 2249 -> 891
Filtered items: 7064 -> 4055
Filtered users: 891 -> 640
Filtered items: 4055 -> 3214
Filtered users: 640 -> 535
Filtered items: 3214 -> 2792
Filtered users: 535 -> 473
Filtered items: 2792 -> 2555
Filtered users: 473 -> 444
Filtered items: 2555 -> 2439
Filtered users: 444 -> 420
Filtered items: 2439 -> 2330
Filtered users: 420 -> 410
Filtered items: 2330 -> 2282
Filtered users: 410 -> 397
Filtered items: 2282 -> 2226
Filtered users: 397 -> 388
Filtered items: 2226 -> 2177
Filtered users: 388 -> 380
Filtered items: 2177 -> 2142
Filtered users: 380 -> 370
Filtered items: 2142 -> 2089
Filtered users: 370 -> 363
Filtered items: 2089 -> 2046
Filtered users: 363 -> 355
Filtered items: 2046 -> 2010
Filtered users: 355 -> 347
Filtered items: 2010 -> 1975
Filtered users: 347 -> 343
Filtered items: 1975 -> 1963
Filtered users: 343 -> 341
Filtered items: 1963 -> 1957
Filtered users: 341 -> 341
Filtered items

In [33]:
#find and remove duplicates and remove date column
duplicates=final_df.duplicated().sum()
if (duplicates>0):
    print("Number of duplicate values:",duplicates)
    final_df.drop_duplicates(inplace=True)

Number of duplicate values: 79


In [34]:
# Get the unique users and items in the final dataframe along with the final
# number of ratings.
final_users = final_df["user"].unique()
final_items = final_df["item"].unique()
final_users_num = len(final_users)
final_items_num = len(final_items)
final_ratings_num = len(final_df)

# Report the final number of unique users and items in the dataset.
print("REDUCED DATASET: {0} number of unique users and {1} of unique items".format(final_users_num,final_items_num))
# Report the final  number of existing ratings in the dataset.
print("REDUCED DATASET: {} total number of existing ratings".format(final_ratings_num))

REDUCED DATASET: 341 number of unique users and 1957 of unique items
REDUCED DATASET: 46196 total number of existing ratings


In [35]:
# We need to reset the users and items ids in order to be able to construct the
# networks of users and items. Users and Items ids should be consecutive integers
# in the [1...final_users_num] and [1...final_items_num].
# Initialy, we need to acquire the sorted versions of the user and item ids.
sorted_final_users = np.sort(final_users)
sorted_final_items = np.sort(final_items)


# Generate the dictionary of final users as a mapping of the following form:
# sorted_final_users --> [0...final_users_num-1]
final_users_dict = dict(zip(sorted_final_users,list(range(0,final_users_num))))
# Generate the dictionary of final items as a mapping of the following form:
# sorted_final_items --> [0...final_items_num-1]
final_items_dict = dict(zip(sorted_final_items,list(range(0,final_items_num))))
# Apply the previously defined dictionary-based maps on the users and item 
# columns of the final dataframe.
final_df["user"] = final_df["user"].map(final_users_dict)
final_df["item"] = final_df["item"].map(final_items_dict)

final_df

,user,item,rating
0,0,312,7
1,31,751,9
2,5,597,6
3,5,590,6
4,0,513,6
...,...,...,...
46270,69,525,5
46271,56,700,10
46272,1,671,6
46273,0,464,6


In [36]:
#set the path for the final dataframe
final_df_path = os.path.join(datafolder,"final_df.pkl")
#check if the file exists
if os.path.exists(final_df_path):
    #load the final dataframe
    final_df = pd.read_pickle(final_df_path)
else:
    #save the final dataframe
    final_df.to_pickle(final_df_path)

In [37]:
item_ratings_count = final_df.groupby("item")["rating"].count().sort_values(ascending=
                                    False).reset_index(name="users_rated")
item_ratings_count.describe()

,item,users_rated
count,1957.000000,1957.000000
mean,978.000000,23.605519
std,565.081558,15.551114
min,0.000000,8.000000
25%,489.000000,13.000000
50%,978.000000,18.000000
75%,1467.000000,29.000000
max,1956.000000,117.000000


In [ ]:
def data_split(dataframe, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    # Get unique items
    items = dataframe['item'].unique()
    
    # Randomly sample items for the test set
    test_items = np.random.choice(items, size=int(len(items) * test_size), replace=False)
    
    # Assign the remaining items to the training set
    train_items = np.setdiff1d(items, test_items)
    
    # Create training and testing DataFrames based on the sampled items
    train_df = dataframe[dataframe['item'].isin(train_items)]
    test_df = dataframe[dataframe['item'].isin(test_items)]
    
    return train_df, test_df

train_df, test_df = data_split(final_df, test_size=0.2, random_state=49)

print(f"Train set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

In [8]:
# Get a grouped version of the final dataframe based on the unique final 
# users.
users_group_df = final_df.groupby("user")
users_group_df["rating"].describe()


NameError: name 'final_df' is not defined

In [39]:
items_group_df=final_df.groupby("item")
a=set(items_group_df.get_group(169)["user"])
a


{3, 4, 44, 48, 71, 86, 101, 143, 191, 212}

In [40]:
# Initialize the adjacency matrix which stores the connection status for each
# pair of users in the recommendation network.
W = np.zeros((final_users_num,final_users_num))
# Initialize the matrix storing the number of commonly rated items for each pair
# of users.
CommonRatings = np.zeros((final_users_num,final_users_num))
# Initialize a the matrix of commmon ratings.
# Matrix W will be of size [final_users_num x final_users_num].
# Let U = {u1,u2,...,un} be the final set of users and I = {i1,i2,...,im} the
# final set of movie items. By considering the function Fi: U --> P(I) where
# P(I) is the powerset of I, Fi(u) returns the subset of items that have been
# rated by user u. In this context, the edge weight between any given pair of
# users (u,v) will be computed as:
#    
#          |Intersection(Fi(u),Fi(v))|
# W(u,v) = ----------------------------
#              |Union(Fi(u),Fi(v)))|

# In order to speed up the construction of the adjacency matrix for the users'
# ratings network, construct a dictionary object that will store a set of the
# rated items for each unique user.
user_items_dict = {}
for user in final_users:
    user_index = final_users_dict[user]
    user_items = set(users_group_df.get_group(user_index)["item"])
    user_items_dict[user_index] = user_items


In [41]:
item_users_dict={}
for item in final_items:
    item_index=final_items_dict[item]
    item_users=set(items_group_df.get_group(item_index)["user"])
    item_users_dict[item_index]=item_users


In [42]:
#for item in item_users_dict:
item=169
if len(item_users_dict[item])!=final_df[final_df["item"]==item].shape[0]:
    print("Error")
    print(item)
    print(item_users_dict[item])
    print(final_df[final_df["item"]==item]["user"].isnotin(item_users_dict[item]))
        
        

In [43]:
itemdf = final_df[final_df['item'] == 1932 ]
itemdf.shape[0]
#len(item_users_dict[87])

80

In [44]:
def precompute_item_split(item_list,split_ratio):
    item_split_dict = {}
    cntr=0
    for item, users in item_list.items():
       users = np.array(list(users))
       np.random.shuffle(users)
       split_idx = int(split_ratio * len(users))  # Fixed 50-50 split
       known,unknown = (users[:split_idx], users[split_idx:])
       itemdf = final_df[final_df['item'] == item]
       ratings_that_exist = itemdf[itemdf['user'].isin(known)]['rating'].to_numpy()
       Rtrue = itemdf[itemdf['user'].isin(unknown)]['rating'].to_numpy()
       if len(Rtrue) !=len(unknown) or len(ratings_that_exist) != len(known):
           print("Error happens here too", item, len(Rtrue), len(unknown), len(ratings_that_exist), len(known))
           cntr+=1
           
       item_split_dict[item] = (known,unknown, ratings_that_exist, Rtrue)
    print("number of errors:",cntr)
    return item_split_dict

item_split_dict = precompute_item_split(item_users_dict, 0.5)
item_split_dict

number of errors: 0


{312: (array([ 51,  49,  13,   3, 274, 317, 113, 312, 222]),
  array([ 55, 223,  25, 276,  28, 139, 228,   0,  80,  66]),
  array([8, 8, 1, 7, 7, 6, 8, 7, 7]),
  array([7, 7, 8, 8, 7, 8, 7, 8, 9, 5])),
 751: (array([ 16, 337,  14,  56, 279, 245,  22,  80, 219, 335, 114,  46, 171,
           4]),
  array([ 48,  31, 172, 338,  70,  54,  19,  61, 222,  97,  10, 180, 329,
          52]),
  array([10, 10,  6, 10,  7,  4,  8,  8,  6,  6,  8,  9,  8,  9]),
  array([ 9,  8,  8, 10,  9,  9,  8,  9, 10,  8,  7,  8,  9,  8])),
 597: (array([128,  70,  72, 233,  38, 168, 229,  42, 226, 138, 327,  94, 300,
          48, 339, 230, 148,  44,  50]),
  array([ 20, 298,  34, 235, 178, 112, 323, 260, 322, 313, 275, 172,   5,
         223, 282, 331,  57, 254, 332]),
  array([ 7, 10,  9,  6, 10,  7,  8,  1,  7,  9,  9,  8,  8,  8,  8,  8,  9,
          5, 10]),
  array([ 6,  8,  8,  8, 10, 10, 10,  9,  7, 10,  9,  9, 10,  4, 10,  9,  9,
          7, 10])),
 590: (array([ 33,  29,   5,  34, 322]),
  array([

In [18]:
item_split_dict[1][3]

array([ 9,  7, 10, 10,  9])

In [19]:
W.sum()

np.float64(0.0)

In [45]:
def calc_error(W, item_list):
    error_sum1 = 0
    error_sum2 = 0
    for item, users_set in item_list.items():
        known, unknown = item_split_dict[item][:2]
        #known, unknown = users[:half], users[half:]
        W_unknown = W[np.ix_(unknown, unknown)]
        W_known_to_unkonown = W[np.ix_(unknown, known)]
        ratings_that_exist = item_split_dict[item][2]
        Rtrue = item_split_dict[item][3]
        if ratings_that_exist.size != W_known_to_unkonown.shape[1]:
            print(f"Mismatch: ratings_that_exist size {ratings_that_exist.size}, W_known_to_unknown columns {W_known_to_unkonown.shape[1]}")
            ratings_that_exist = ratings_that_exist[:W_known_to_unkonown.shape[1]]
            #print(ratings_that_exist)
            #print(W_known_to_unkonown)
            #break
            error_sum1+=1
            
        if Rtrue.size != W_unknown.shape[0]:
            print(f"Mismatch: Rtrue size {Rtrue.size}, W_unknown rows {W_unknown.shape[0]}")
            Rtrue = Rtrue[:W_unknown.shape[0]]
            error_sum2+=1
    print(error_sum1+error_sum2)

calc_error(W, item_split_dict)

0


In [46]:
# Sort the users' items dictionary by the user_index.
user_ids = list(user_items_dict.keys())
user_ids.sort()
user_ids


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [47]:
# Generate the sorted version of the dictionary.
user_items_dict = {user_index:user_items_dict[user_index] for user_index in user_ids}
user_items_dict


{0: {157,
  158,
  179,
  180,
  183,
  185,
  190,
  205,
  218,
  219,
  242,
  245,
  249,
  312,
  330,
  331,
  350,
  377,
  378,
  392,
  393,
  401,
  404,
  409,
  418,
  419,
  422,
  426,
  430,
  433,
  443,
  445,
  450,
  453,
  458,
  462,
  464,
  467,
  471,
  479,
  483,
  486,
  511,
  513,
  519,
  531,
  540,
  545,
  554,
  555,
  557,
  561,
  567,
  573,
  576,
  584,
  585,
  590,
  599,
  604,
  606,
  613,
  618,
  619,
  620,
  623,
  624,
  628,
  635,
  637,
  640,
  644,
  646,
  647,
  648,
  649,
  651,
  652,
  653,
  654,
  659,
  676,
  681,
  683,
  688,
  690,
  691,
  692,
  693,
  698,
  703,
  708,
  711,
  714,
  715,
  717,
  719,
  725,
  727,
  728,
  741,
  742,
  744,
  746,
  753,
  754,
  755,
  757,
  758,
  759,
  760,
  764,
  766,
  770,
  771,
  773,
  775,
  777,
  783,
  787,
  793,
  794,
  796,
  800,
  812,
  814,
  815,
  816,
  820,
  821,
  825,
  833,
  851,
  861,
  868,
  869,
  878,
  884,
  888,
  897,
  906,
  946,
  1

In [ ]:
#testing
for user, items in user_items_dict.items():
    print(items)
    break


In [48]:
# Set the pickle file that will store the graph adjacency matrix W.
adjacency_numpy_file = os.path.join(datafolder,"W.npy")
common_ratings_numpy_file = os.path.join(datafolder,"CommonRatings.npy")
# Check the existence of the previously defined numpy file.
if os.path.exists(adjacency_numpy_file):
    # Load the numpy file.
    W = np.load(adjacency_numpy_file,allow_pickle=True)
    CommonRatings = np.load(common_ratings_numpy_file,allow_pickle=True)
else:
    # Loop through the rectangular grid of users that index the elements of matrix 
    # W.
    for source_user in user_items_dict.keys():
        for target_user in user_items_dict.keys():
            print("Pricessing users' pair ({0},{1})".format(source_user,target_user))
            intersection_items = user_items_dict[source_user].intersection(user_items_dict[target_user])
            union_items = user_items_dict[source_user].union(user_items_dict[target_user])
            W[source_user,target_user] = len(intersection_items) / len(union_items)
            CommonRatings[source_user,target_user] = len(intersection_items)
    # Save adjacency matrix to prespecified pickle file.
    np.save(adjacency_numpy_file,W)
    np.save(common_ratings_numpy_file,CommonRatings)

Pricessing users' pair (0,0)
Pricessing users' pair (0,1)
Pricessing users' pair (0,2)
Pricessing users' pair (0,3)
Pricessing users' pair (0,4)
Pricessing users' pair (0,5)
Pricessing users' pair (0,6)
Pricessing users' pair (0,7)
Pricessing users' pair (0,8)
Pricessing users' pair (0,9)
Pricessing users' pair (0,10)
Pricessing users' pair (0,11)
Pricessing users' pair (0,12)
Pricessing users' pair (0,13)
Pricessing users' pair (0,14)
Pricessing users' pair (0,15)
Pricessing users' pair (0,16)
Pricessing users' pair (0,17)
Pricessing users' pair (0,18)
Pricessing users' pair (0,19)
Pricessing users' pair (0,20)
Pricessing users' pair (0,21)
Pricessing users' pair (0,22)
Pricessing users' pair (0,23)
Pricessing users' pair (0,24)
Pricessing users' pair (0,25)
Pricessing users' pair (0,26)
Pricessing users' pair (0,27)
Pricessing users' pair (0,28)
Pricessing users' pair (0,29)
Pricessing users' pair (0,30)
Pricessing users' pair (0,31)
Pricessing users' pair (0,32)
Pricessing users' pa

In [ ]:
# Save the dictionary to a pickle file
output_pickle_file = os.path.normpath(os.path.join(datafolder, "item_users_dict.pkl"))
with open(output_pickle_file, 'wb') as f:
    pickle.dump(item_users_dict, f)

In [27]:
W.sum()

np.float64(6241.7326236031795)

In [10]:

# Read the CSV files
results_df = pd.read_csv("ga_experiments.csv")
results_df1 = pd.read_csv("ga_experiments1.csv")

# Concatenate the dataframes
combined_df = pd.concat([results_df, results_df1], ignore_index=True)

# Reset the index
combined_df.reset_index(drop=True, inplace=True)

combined_df

,population_size,generations,sample_ratio,similarity_penalty,elitism,lamda_reg,split_ratio,noise_scale,mutation_rate,scale,best_fitness,avg_fitness,timestamp
0,50,50,0.4,0.25,10,0.01,0.50,0.2,0.10,0.1,33.359651,33.866149,2025-01-30 20:33:25
1,50,50,0.4,0.25,10,0.05,0.50,0.2,0.10,0.1,33.544466,33.996070,2025-01-30 20:36:49
2,30,40,0.4,0.25,5,0.01,0.80,0.2,0.10,0.1,32.798641,33.418512,2025-01-30 20:40:21
3,30,40,0.3,0.25,5,0.10,0.80,0.2,0.10,0.1,31.697887,32.484707,2025-01-30 20:54:26
4,30,40,0.3,0.25,5,0.30,0.80,0.2,0.10,0.1,32.501605,33.081095,2025-01-30 21:44:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,20,40,0.4,0.05,2,0.33,0.23,0.2,0.49,0.1,33.172409,33.631685,2025-01-31 00:34:36
129,20,40,0.4,0.05,2,0.33,0.23,0.2,0.51,0.1,33.172934,33.637344,2025-01-31 00:36:45
130,20,150,0.3,0.05,2,0.33,0.23,0.2,0.03,0.1,33.127834,33.368920,2025-01-31 00:45:50
131,20,150,0.3,0.05,2,0.33,0.23,0.2,0.03,0.1,32.806848,33.039127,2025-01-31 01:05:47


In [11]:
sorted_df = combined_df.sort_values(by="best_fitness", ascending=True)
sorted_df

,population_size,generations,sample_ratio,similarity_penalty,elitism,lamda_reg,split_ratio,noise_scale,mutation_rate,scale,best_fitness,avg_fitness,timestamp
29,100,500,0.6,0.25,5,0.01,0.80,0.20,0.01,0.1,3.485530,3.496258,2025-02-03 23:39:39
30,100,500,0.6,0.10,5,0.50,0.80,0.30,0.50,0.1,3.885733,4.009497,2025-02-03 23:47:40
25,25,500,0.6,0.25,2,0.30,0.50,0.05,0.30,0.1,4.327443,4.356099,2025-01-31 01:50:55
26,25,500,0.6,0.20,12,0.30,0.50,0.90,0.30,0.1,4.354826,4.372229,2025-01-31 01:54:26
23,25,30,0.6,0.25,2,0.30,0.50,0.05,0.30,0.1,5.048760,5.076467,2025-01-31 01:46:51
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,20,40,0.4,0.05,2,0.33,0.27,0.20,0.10,0.1,33.536611,34.044042,2025-01-30 23:02:14
1,50,50,0.4,0.25,10,0.05,0.50,0.20,0.10,0.1,33.544466,33.996070,2025-01-30 20:36:49
121,20,40,0.4,0.05,2,0.33,0.23,0.20,0.33,0.1,33.573782,33.962401,2025-01-31 00:19:14
22,25,30,0.6,0.25,2,0.30,0.50,0.05,0.30,0.1,33.599354,33.937279,2025-01-31 01:45:29


In [ ]:
#other vtratons of initail pop
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh

def create_spectral_population(size, user_item_matrix):
    # Compute affinity matrix using cosine similarity
    affinity = cosine_similarity(user_item_matrix)
    # Compute graph Laplacian
    L = laplacian(affinity, normed=True)
    # Compute spectral embedding
    _, eigenvectors = eigsh(L, k=10, which='SM')  # 10 smallest eigenvectors
    spectral_embedding = eigenvectors[:, 1:]      # Skip first trivial eigenvector
    # Create population
    population = []
    for _ in range(size):
        noise = np.random.normal(scale=0.1, spectral_embedding.shape)
        heuristic_matrix = spectral_embedding @ spectral_embedding.T + noise
        heuristic_matrix = np.clip(heuristic_matrix, 0.001, 0.999)
        population.append(heuristic_matrix)
    return population

In [ ]:
from scipy.sparse.linalg import svds
from sklearn.decomposition import NMF

def create_svd_population(size, user_item_matrix, rank=10):
    population = []
    # Compute SVD
    U, S, Vt = svds(user_item_matrix, k=rank)
    W_svd = U @ np.diag(S)  # Low-rank user features
    for _ in range(size):
        # Perturb SVD-based matrix
        noise = np.random.normal(scale=0.1, size=W_svd.shape)
        heuristic_matrix = W_svd @ W_svd.T + noise  # Symmetric by construction
        heuristic_matrix = np.clip(heuristic_matrix, 0.001, 0.999)
        population.append(heuristic_matrix)
    return population

def create_nmf_population(size, user_item_matrix, rank=10):
    model = NMF(n_components=rank)
    W_nmf = model.fit_transform(user_item_matrix)  # User-factor matrix
    H_nmf = model.components_                     # Factor-item matrix
    return [W_nmf @ W_nmf.T for _ in range(size)]  # Symmetric affinity matrices

In [ ]:
from scipy.spatial.distance import pdist, squareform
from sklearn.gaussian_process.kernels import RBF

def create_kernel_population(size, user_item_matrix):
    # Compute RBF kernel
    kernel = RBF(length_scale=1.0)
    K = kernel(user_item_matrix)
    # Add noise for diversity
    population = []
    for _ in range(size):
        noise = np.random.normal(scale=0.1, size=K.shape)
        heuristic_matrix = K + noise
        heuristic_matrix = (heuristic_matrix + heuristic_matrix.T) / 2  # Enforce symmetry
        population.append(heuristic_matrix)
    return population

In [ ]:
def create_hybrid_population(size, user_item_matrix):
    size_per_method = size // 4
    populations = [
        create_nmf_population(size_per_method, user_item_matrix),
        create_spectral_population(size_per_method, user_item_matrix),
        create_svd_population(size_per_method, user_item_matrix),
        create_nmf_population(size_per_method, user_item_matrix)
    ]
    return populations